# Polymarket Data Collection Notebook

This notebook demonstrates the data collection process for the Polymarket Alpha Research project.

**Research Question**: Can machine learning algorithms, trained on publicly available blockchain transaction data, accurately identify informed traders in decentralized prediction markets?

## Overview

This notebook covers:
1. Environment setup and dependencies
2. Data collection from Polymarket public APIs
3. Market resolution data fetching
4. Data storage and validation
5. Sample data exploration

**Data Source**: Polymarket Gamma API & Data API (public REST endpoints)
**Time Period**: 2024-2025
**Collection Method**: REST API polling with rate limiting

## 1. Environment Setup

First, install required dependencies:

In [ ]:
# Install required packages
# !pip install requests pandas loguru python-dotenv sqlalchemy

In [ ]:
import sys
import os
import json
import time
import requests
import datetime
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Optional

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")

## 2. Configuration

Define API endpoints and collection parameters:

In [ ]:
# API Configuration
GAMMA_API = "https://gamma-api.polymarket.com"
DATA_API = "https://data-api.polymarket.com"

# Collection Parameters
MAX_TRADES = 5000  # Maximum trades to collect for demo
PAGE_SIZE = 100    # API pagination size
RATE_LIMIT_DELAY = 0.25  # Seconds between requests

# Data Paths
DATA_DIR = project_root / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Create directories
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Max trades to collect: {MAX_TRADES}")

## 3. Data Collection Implementation

### 3.1 Fetch Global Trade Feed

Collect transaction data from Polymarket's public trades API:

In [ ]:
def fetch_global_trades(max_total: int = 5000) -> List[Dict]:
    """
    Fetch trades from Polymarket data API.
    
    Args:
        max_total: Maximum number of trades to collect
    
    Returns:
        List of trade dictionaries with transaction details
    """
    session = requests.Session()
    session.headers.update({
        "User-Agent": "PolymarketAlphaResearch/3.0",
        "Accept": "application/json",
    })
    
    url = f"{DATA_API}/trades"
    all_trades = []
    seen_tx_hashes = set()
    offset = 0
    empty_pages = 0
    
    print(f"Fetching up to {max_total} trades...")
    
    while len(all_trades) < max_total:
        try:
            resp = session.get(
                url, 
                params={"limit": PAGE_SIZE, "offset": offset}, 
                timeout=15
            )
            resp.raise_for_status()
            trades = resp.json()
        except requests.exceptions.RequestException as e:
            print(f"Request failed at offset {offset}: {e}")
            empty_pages += 1
            if empty_pages >= 3:
                break
            time.sleep(1)
            offset += PAGE_SIZE
            continue
        
        if not trades or not isinstance(trades, list) or len(trades) == 0:
            empty_pages += 1
            if empty_pages >= 3:
                break
            offset += PAGE_SIZE
            time.sleep(RATE_LIMIT_DELAY)
            continue
        
        empty_pages = 0
        
        for t in trades:
            tx_hash = t.get("transactionHash", "")
            if tx_hash and tx_hash not in seen_tx_hashes:
                seen_tx_hashes.add(tx_hash)
                
                # Convert timestamp
                ts_unix = t.get("timestamp", 0)
                try:
                    ts_iso = datetime.datetime.fromtimestamp(
                        int(ts_unix), tz=datetime.timezone.utc
                    ).strftime("%Y-%m-%dT%H:%M:%SZ")
                except:
                    ts_iso = ""
                
                all_trades.append({
                    "transaction_id": tx_hash,
                    "address": t.get("proxyWallet", ""),
                    "market_id": t.get("conditionId", ""),
                    "side": t.get("side", ""),
                    "amount": float(t.get("size", 0)),
                    "price": float(t.get("price", 0)),
                    "timestamp": ts_iso,
                    "outcome": t.get("outcome", ""),
                    "market_title": t.get("title", ""),
                })
        
        if len(all_trades) % 500 < PAGE_SIZE:
            print(f"  Progress: {len(all_trades)} trades collected")
        
        offset += PAGE_SIZE
        time.sleep(RATE_LIMIT_DELAY)
    
    print(f"\nCollection complete: {len(all_trades)} unique trades")
    return all_trades

In [ ]:
# Execute trade collection (this may take a few minutes)
# Note: Uncomment to run actual data collection
# trades = fetch_global_trades(max_total=1000)

# For demonstration, we'll show the structure
print("Trade data structure (example):")
sample_trade = {
    "transaction_id": "0xabc123...",
    "address": "0xwallet...",
    "market_id": "0xmarket...",
    "side": "BUY",  # or 'SELL'
    "amount": 100.0,  # USDC
    "price": 0.65,    # price per share
    "timestamp": "2024-03-15T14:30:00Z",
    "outcome": "Yes",
    "market_title": "Will Bitcoin exceed $100k by Dec 2024?"
}
print(json.dumps(sample_trade, indent=2))

### 3.2 Fetch Market Resolutions

Collect market outcome data (which side won/lost):

In [ ]:
def fetch_market_resolution(condition_id: str) -> Optional[str]:
    """
    Fetch resolution for a single market.
    
    Args:
        condition_id: Market condition ID
    
    Returns:
        Winning outcome string or None if unresolved
    """
    try:
        session = requests.Session()
        resp = session.get(
            f"{GAMMA_API}/markets",
            params={"conditionId": condition_id, "limit": 1},
            timeout=10
        )
        resp.raise_for_status()
        markets = resp.json()
        
        if markets and isinstance(markets, list) and len(markets) > 0:
            m = markets[0]
            outcomes = json.loads(m.get("outcomes", "[]"))
            outcome_prices = json.loads(m.get("outcomePrices", "[]"))
            
            if outcome_prices and outcomes:
                prices_float = [float(p) for p in outcome_prices]
                winner_idx = prices_float.index(max(prices_float))
                if winner_idx < len(outcomes):
                    if not m.get("closed", False):
                        return "__OPEN__"
                    return outcomes[winner_idx]
    except Exception as e:
        print(f"Failed to fetch resolution for {condition_id[:20]}: {e}")
    return None


def fetch_all_resolutions(condition_ids: List[str], max_workers: int = 5) -> Dict[str, str]:
    """
    Fetch resolutions for multiple markets concurrently.
    
    Args:
        condition_ids: List of market condition IDs
        max_workers: Number of concurrent threads
    
    Returns:
        Dictionary mapping market_id to resolution
    """
    resolutions = {}
    unique_cids = list(set(condition_ids))
    
    print(f"Fetching resolutions for {len(unique_cids)} markets...")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_cid = {
            executor.submit(fetch_market_resolution, cid): cid
            for cid in unique_cids
        }
        
        for i, future in enumerate(as_completed(future_to_cid)):
            cid = future_to_cid[future]
            resolution = future.result()
            if resolution:
                resolutions[cid] = resolution
            
            if (i + 1) % 10 == 0:
                print(f"  Resolved {i + 1}/{len(unique_cids)} markets")
    
    print(f"Got resolutions for {len(resolutions)} markets")
    return resolutions

### 3.3 Save Data

Store collected data to CSV files:

In [ ]:
def save_trades_to_csv(trades: List[Dict], filepath: Path):
    """Save trade data to CSV."""
    df = pd.DataFrame(trades)
    df.to_csv(filepath, index=False)
    print(f"Saved {len(df)} trades to {filepath}")
    return df


def save_resolutions_to_csv(resolutions: Dict[str, str], trades: List[Dict], filepath: Path):
    """Save market resolutions to CSV."""
    records = []
    for cid, resolution in resolutions.items():
        sample = next((t for t in trades if t.get("market_id") == cid), {})
        records.append({
            "market_id": cid,
            "question": sample.get("market_title", ""),
            "resolution": resolution,
        })
    
    df = pd.DataFrame(records)
    df.to_csv(filepath, index=False)
    print(f"Saved {len(df)} market resolutions to {filepath}")
    return df

## 4. Run Full Collection

Execute the complete data collection pipeline:

In [ ]:
def run_full_collection(max_trades: int = 1000):
    """
    Run the complete data collection pipeline.
    
    Args:
        max_trades: Maximum trades to collect
    """
    print("=" * 60)
    print("STARTING DATA COLLECTION PIPELINE")
    print("=" * 60)
    
    # Step 1: Collect trades
    trades = fetch_global_trades(max_total=max_trades)
    
    if not trades:
        print("No trades collected. Exiting.")
        return None
    
    # Step 2: Collect market resolutions
    unique_markets = list(set(t["market_id"] for t in trades if t["market_id"]))
    resolutions = fetch_all_resolutions(unique_markets)
    
    # Step 3: Save data
    trades_path = PROCESSED_DIR / "trades_collected.csv"
    resolutions_path = PROCESSED_DIR / "market_resolutions.csv"
    
    trades_df = save_trades_to_csv(trades, trades_path)
    resolutions_df = save_resolutions_to_csv(resolutions, trades, resolutions_path)
    
    # Summary
    print("\n" + "=" * 60)
    print("COLLECTION SUMMARY")
    print("=" * 60)
    print(f"Total trades: {len(trades)}")
    print(f"Unique wallets: {len(set(t['address'] for t in trades))}")
    print(f"Unique markets: {len(unique_markets)}")
    print(f"Resolved markets: {sum(1 for v in resolutions.values() if v != '__OPEN__')}")
    print(f"\nOutput files:")
    print(f"  - {trades_path}")
    print(f"  - {resolutions_path}")
    
    return trades_df, resolutions_df


# Run collection (uncomment to execute)
# trades_df, resolutions_df = run_full_collection(max_trades=1000)

## 5. Load and Explore Sample Data

If you have existing data, load and explore it:

In [ ]:
# Load existing data if available
trades_sample_path = PROCESSED_DIR / "trades_collected.csv"

if trades_sample_path.exists():
    df = pd.read_csv(trades_sample_path)
    print(f"Loaded {len(df)} trades")
    print("\nFirst few rows:")
    display(df.head())
    print("\nData summary:")
    print(df.describe())
else:
    print("No existing data found. Run collection first.")

## 6. Data Validation

Check data quality and completeness:

In [ ]:
def validate_data(df: pd.DataFrame):
    """
    Validate collected data quality.
    
    Args:
        df: Trade dataframe
    """
    print("Data Validation Report")
    print("-" * 40)
    
    # Check for missing values
    missing = df.isnull().sum()
    print(f"Missing values per column:")
    print(missing[missing > 0] if missing.sum() > 0 else "  None")
    
    # Check for duplicates
    dups = df.duplicated(subset=["transaction_id"]).sum()
    print(f"\nDuplicate transactions: {dups}")
    
    # Check value ranges
    print(f"\nAmount range: ${df['amount'].min():.2f} - ${df['amount'].max():.2f}")
    print(f"Price range: {df['price'].min():.2f} - {df['price'].max():.2f}")
    
    # Check date range
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    
    # Check side distribution
    print(f"\nSide distribution:")
    print(df['side'].value_counts())


# Run validation if data exists
if trades_sample_path.exists():
    df = pd.read_csv(trades_sample_path)
    validate_data(df)

## 7. Export Sample for GitHub

Create a sample dataset for repository upload:

In [ ]:
def export_sample_for_github(full_data_path: Path, sample_path: Path, n: int = 1000):
    """
    Export first n rows as sample for GitHub.
    
    Args:
        full_data_path: Path to full dataset
        sample_path: Path to save sample
        n: Number of rows to include
    """
    if not full_data_path.exists():
        print(f"Full data not found at {full_data_path}")
        return
    
    df = pd.read_csv(full_data_path)
    sample = df.head(n)
    sample.to_csv(sample_path, index=False)
    
    print(f"Exported {len(sample)} rows to {sample_path}")
    print(f"File size: {sample_path.stat().st_size / 1024:.1f} KB")


# Define paths
DATA_DIR_ROOT = project_root / "data"
sample_output = DATA_DIR_ROOT / "polymarket_sample.csv"

# If you have a full dataset, export sample
# full_data = PROCESSED_DIR / "trades_collected.csv"
# export_sample_for_github(full_data, sample_output)

---

## Summary

This notebook demonstrates:

1. **Data Source**: Polymarket public REST APIs (no authentication required)
2. **Collection Method**: Pagination with rate limiting and concurrent resolution fetching
3. **Output Format**: CSV files with transaction and market resolution data
4. **Data Quality**: Built-in validation for duplicates, missing values, and range checks

### Output Files

- `data/processed/trades_collected.csv`: Transaction-level trade data
- `data/processed/market_resolutions.csv`: Market outcome resolutions
- `data/polymarket_sample.csv`: Sample for GitHub repository (first 1000 rows)

### Running the Collection

To run actual data collection:
1. Uncomment the collection cells above
2. Run cells sequentially
3. Check output in `data/processed/` directory